# Random Forest

**Sessão 2 · Classificação · Parte 2**

*Inteligência Artificial e Aprendizagem de Máquina · FECAP · 2026/02*

## Carregando os animais

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

animais = json.load(open("animais_completo.json", encoding="utf-8"))
df = pd.DataFrame(animais)

features = ["voa", "nada", "bota_ovos", "tem_quatro_patas",
            "vive_na_agua", "e_carnivoro", "tem_cauda", "e_domestico"]
X = df[features]
y = df["tem_pelos"]

## Discordância genuína entre árvores

Rodando 15 árvores diferentes (cada uma vendo uma amostra aleatória dos 85 animais), pedimos a todas pra classificar a mesma aranha.

In [ ]:
aranha = X.iloc[[df[df['nome']=='Aranha'].index[0]]]

rng = np.random.default_rng(0)
previsoes = []
for i in range(15):
    idx = rng.choice(len(X), size=len(X), replace=True)
    arvore = DecisionTreeClassifier(max_depth=3, random_state=i)
    arvore.fit(X.iloc[idx], y.iloc[idx])
    previsoes.append(bool(arvore.predict(aranha)[0]))

print(previsoes)
print(f"{sum(previsoes)} de 15 disseram True")

**Não é erro de código — é a instabilidade real de uma árvore única, na cara.**

## Como o Random Forest resolve isso

Em vez de confiar na opinião de 1 árvore, o Random Forest treina centenas delas e agrega os votos.

In [ ]:
floresta = RandomForestClassifier(n_estimators=100, random_state=42)
floresta.fit(X, y)

prob = floresta.predict_proba(aranha)
print(f"Probabilidade de ter pelo: {prob[0][1]:.1%}")

## De onde vem a diversidade entre as árvores

**Bootstrap**: cada árvore é treinada numa amostra aleatória dos animais (com reposição).

**max_features**: em cada divisão, cada árvore só pode escolher entre um subconjunto aleatório das colunas.

## O papel do n_estimators

Quantidade de árvores na floresta. Diferente de max_depth ou K, mais árvores quase nunca piora — só fica mais lento pra treinar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=5, stratify=y
)

valores_n = [1, 5, 10, 25, 50, 75, 100, 150, 200]
acuracias = []
for n in valores_n:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train, y_train)
    acuracias.append(accuracy_score(y_test, m.predict(X_test)))

plt.plot(valores_n, acuracias, marker="o")
plt.xlabel("n_estimators")
plt.ylabel("Acuracia")
plt.show()

## O fluxo no Scikit-Learn

Mesma gramática de sempre: `RandomForestClassifier(n_estimators=N)`, `fit()`, `predict()`.

In [ ]:
modelo_final = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_final.fit(X_train, y_train)

pred = modelo_final.predict(X_test)
print(f"Acuracia: {accuracy_score(y_test, pred):.2%}")

## Importância das features no Random Forest

Mesma ideia da árvore única, mas mais confiável — é uma média entre centenas de árvores, não o julgamento de uma só.

In [ ]:
importancias = pd.Series(
    modelo_final.feature_importances_, index=features
).sort_values()

importancias.plot(kind="barh")
plt.xlabel("Importancia")
plt.show()

---
## Fechando Random Forest

Vimos como árvores individuais discordam entre si, como o Random Forest resolve isso agregando votos, as duas fontes de aleatoriedade (bootstrap + max_features), e como interpretar a importância das features de forma mais confiável.